# Train TotalCharges Model

In [0]:
%pip install feature_engine
%pip install xgboost lightgbm catboost
%pip install seaborn

In [0]:
# Table
tb_abt = 'raw_abt1_totalcharges'

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.sklearn
from sklearn import model_selection
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import logging

In [0]:
# Suppress the noisy MLflow tracking context registry warnings
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)

# Set the registry to Unity Catalog
mlflow.set_experiment(experiment_id=1316272881415465)




In [0]:

df = spark.table(f'workspace.default.{tb_abt}').toPandas()

X = df.drop(columns=['customerID', 'TotalCharges']).fillna(0).astype(float)
y = df["TotalCharges"].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Size Train dataset: {y_train.shape[0]} ({y_train.shape[0]/y.shape[0]*100:.2f}%). Target mean: ({y_train.mean():.2f})')

print(f'Size Test dataset: {y_test.shape[0]} ({y_test.shape[0]/y.shape[0]*100:.2f}%). Target mean: ({y_test.mean():.2f})')

In [0]:
df.isnull().sum()

## Reusable Model Template

In [0]:
# REUSABLE MODEL TEMPLATE
def train_and_log(
    model_name,
    estimator,
    param_grid=None,          # None -> plain fit; dict/list -> GridSearchCV
    cv=5,
    scoring='neg_mean_squared_error',
    X_train=None, y_train=None, X_test=None, y_test=None,
    max_tuning_runs=20,
    feature_list=None,
):
    # Autolog: logs params, metrics, model signature, and GridSearchCV child runs
    mlflow.sklearn.autolog(exclusive=False, max_tuning_runs=max_tuning_runs)

    with mlflow.start_run(run_name=model_name) as run:
        # ---- Log feature count up front ----
        if feature_list is not None:
            mlflow.log_param('n_features', len(feature_list))
        else:
            mlflow.log_param('n_features', X_train.shape[1])

        # ---- Train ----
        if param_grid is not None:
            model = GridSearchCV(
                estimator=estimator,
                param_grid=param_grid,
                cv=cv,
                scoring=scoring,
                n_jobs=-1,
            )
            model.fit(X_train, y_train)
            best_model = model.best_estimator_
            mlflow.log_params(model.best_params_)
            mlflow.log_metric('best_cv_score', model.best_score_)
            print(f'Best params: {model.best_params_}')
        else:
            model = estimator
            model.fit(X_train, y_train)
            best_model = model

        # ---- Predict + metrics (using the best estimator) ----
        train_preds = best_model.predict(X_train)
        test_preds = best_model.predict(X_test)

        mlflow.log_metrics({
            'train_rmse': np.sqrt(mean_squared_error(y_train, train_preds)),
            'train_mae': mean_absolute_error(y_train, train_preds),
            'train_r2': r2_score(y_train, train_preds),
            'train_mape': mean_absolute_percentage_error(y_train, train_preds),
            'test_rmse': np.sqrt(mean_squared_error(y_test, test_preds)),
            'test_mae': mean_absolute_error(y_test, test_preds),
            'test_r2': r2_score(y_test, test_preds),
            'test_mape': mean_absolute_percentage_error(y_test, test_preds),
        })

        # ---- Register best model ----
        mlflow.sklearn.log_model(
            sk_model=best_model,
            name='model',
            registered_model_name=f'workspace.default.{model_name}',
            input_example=X_train.iloc[[0]],
        )

        print(f'Model: {model_name} done!')

### Training  Models

- OLS baseline
- Plain Ridge
- Ridge w/ Tunning (Grid Search & Cross Validation)

In [0]:
# OLS baseline
train_and_log(
    model_name='total_charges_baseline_ols',
    estimator=LinearRegression(),
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
)


In [0]:
# Plain Ridge (no tuning)
train_and_log(
    model_name='total_charges_ridge',
    estimator=Ridge(alpha=0.1),
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
)

In [0]:
# Ridge with GridSearchCV
train_and_log(
    model_name='total_charges_ridge_tuned',
    estimator=Ridge(),
    param_grid={'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]},
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
)